# Day 7. Transformer

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Scaled Dot-Product Attention 구현

In [2]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        """
        Q: (batch, L_q, d_k)
        K: (batch, L_k, d_k)
        V: (batch, L_k, d_v)
        mask: (batch, L_q, L_k) or (1, L_q, L_k), 0 또는 -inf 마스킹용
        """
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        output = torch.matmul(attn, V)
        return output, attn


## Multi-Head Attention 구현 (batch_first = True 가정)

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(dropout)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        """
        x: (B, L, d_model)  -- self-attention 입력
        mask: (B, L, L) or (1, L, L)
        """
        B, L, _ = x.size()

        residual = x

        # 1) Linear projections
        Q = self.W_q(x)  # (B, L, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2) head 나누기
        # (B, L, num_heads, d_k) -> (B, num_heads, L, d_k)
        def split_heads(t):
            return t.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

        Q = split_heads(Q)
        K = split_heads(K)
        V = split_heads(V)

        if mask is not None:
            # mask: (B, L, L) -> (B, 1, L, L) for broadcasting
            mask = mask.unsqueeze(1)

        # 3) attention
        context, attn = self.attention(Q, K, V, mask=mask)  # context: (B, num_heads, L, d_k)

        # 4) concat heads
        context = context.transpose(1, 2).contiguous().view(B, L, self.d_model)

        # 5) final linear + residual + layer norm
        output = self.fc(context)
        output = self.dropout(output)
        output = self.layer_norm(output + residual)

        return output, attn


## Position-wise Feed-Forward Network

In [4]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.layer_norm(x + residual)
        return x


## Transformer Encoder Block

In [5]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

    def forward(self, x, src_mask=None):
        """
        x: (B, L, d_model)
        src_mask: (B, L, L) or (1, L, L)
        """
        x, attn = self.self_attn(x, mask=src_mask)
        x = self.ffn(x)
        return x, attn


## Transformer Encoder Block 간단 테스트

In [6]:
batch_size = 2
seq_len = 5
d_model = 16

x = torch.randn(batch_size, seq_len, d_model).to(device)  # toy input

block = TransformerEncoderBlock(
    d_model=d_model, num_heads=4, d_ff=64, dropout=0.1
).to(device)

out, attn = block(x)

print("Input shape :", x.shape)       # (2, 5, 16)
print("Output shape:", out.shape)     # (2, 5, 16)
print("Attention shape:", attn.shape) # (2, num_heads, L, L)


Input shape : torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention shape: torch.Size([2, 4, 5, 5])


## Sin-Cos Positional Encoding

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: (B, L, d_model)
        """
        L = x.size(1)
        x = x + self.pe[:, :L]
        return x


## Encoder-Decoder Transformer (PyTorch 기본 모듈 활용)

In [8]:
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_tokens_src: int,
        num_tokens_tgt: int,
        d_model: int = 128,
        nhead: int = 4,
        num_encoder_layers: int = 2,
        num_decoder_layers: int = 2,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
        max_len: int = 100,
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embed = nn.Embedding(num_tokens_src, d_model)
        self.tgt_embed = nn.Embedding(num_tokens_tgt, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_decoder_layers,
        )

        self.generator = nn.Linear(d_model, num_tokens_tgt)

    def make_src_mask(self, src):
        src_pad_mask = (src == 0)
        return src_pad_mask

    def make_tgt_mask(self, tgt):
        L = tgt.size(1)
        mask = torch.triu(torch.ones(L, L), diagonal=1).bool()
        return mask

    def forward(self, src, tgt):
        B, L_src = src.shape
        B, L_tgt = tgt.shape

        src_mask = None
        tgt_mask = self.make_tgt_mask(tgt).to(src.device)
        src_key_padding_mask = self.make_src_mask(src)
        tgt_key_padding_mask = (tgt == 0)

        # ↓↓↓ math.sqrt → np.sqrt 로 변경
        src_emb = self.positional_encoding(
            self.src_embed(src) * np.sqrt(self.d_model)
        )
        tgt_emb = self.positional_encoding(
            self.tgt_embed(tgt) * np.sqrt(self.d_model)
        )

        memory = self.encoder(
            src_emb,
            mask=src_mask,
            src_key_padding_mask=src_key_padding_mask,
        )

        out = self.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = self.generator(out)
        return logits


In [9]:
vocab_src = 50
vocab_tgt = 50
model = Seq2SeqTransformer(
    num_tokens_src=vocab_src,
    num_tokens_tgt=vocab_tgt,
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    dim_feedforward=512,
    dropout=0.1,
    max_len=50,
).to(device)

B = 4
L_src = 7
L_tgt = 6

src = torch.randint(1, vocab_src, (B, L_src)).to(device)
tgt = torch.randint(1, vocab_tgt, (B, L_tgt)).to(device)

logits = model(src, tgt)
print("src shape:", src.shape)      # (B, L_src)
print("tgt shape:", tgt.shape)      # (B, L_tgt)
print("logits shape:", logits.shape) # (B, L_tgt, vocab_tgt)


src shape: torch.Size([4, 7])
tgt shape: torch.Size([4, 6])
logits shape: torch.Size([4, 6, 50])


## Toy Dataset (시퀀스 뒤집기) 생성

In [10]:


import random

PAD = 0
SOS = 1   # start of sequence
EOS = 2   # end of sequence (필요하면 사용)

vocab_size = 60  # 0~59, 0은 PAD, 1,2는 SOS/EOS, 3~59가 숫자 토큰

def generate_example(min_len=3, max_len=8):
    seq_len = random.randint(min_len, max_len)
    seq = [random.randint(3, vocab_size - 1) for _ in range(seq_len)]
    src = seq
    tgt = list(reversed(seq))
    return src, tgt

def pad_sequence(seq, max_len, pad_value=PAD):
    return seq + [pad_value] * (max_len - len(seq))

def make_batch(batch_size, min_len=3, max_len=8):
    src_list, tgt_in_list, tgt_out_list = [], [], []

    for _ in range(batch_size):
        src, tgt = generate_example(min_len, max_len)

        # 디코더 입력: [SOS] + target
        tgt_in = [SOS] + tgt
        # 디코더 정답: target + [EOS]
        tgt_out = tgt + [EOS]

        src_list.append(src)
        tgt_in_list.append(tgt_in)
        tgt_out_list.append(tgt_out)

    # 길이 맞추기
    max_src_len = max(len(s) for s in src_list)
    max_tgt_len = max(len(t) for t in tgt_in_list)

    src_batch = torch.tensor(
        [pad_sequence(s, max_src_len, PAD) for s in src_list],
        dtype=torch.long,
    )
    tgt_in_batch = torch.tensor(
        [pad_sequence(t, max_tgt_len, PAD) for t in tgt_in_list],
        dtype=torch.long,
    )
    tgt_out_batch = torch.tensor(
        [pad_sequence(t, max_tgt_len, PAD) for t in tgt_out_list],
        dtype=torch.long,
    )

    return src_batch.to(device), tgt_in_batch.to(device), tgt_out_batch.to(device)

# 테스트
src_b, tgt_in_b, tgt_out_b = make_batch(2)
print("src_b:", src_b)
print("tgt_in_b:", tgt_in_b)
print("tgt_out_b:", tgt_out_b)


src_b: tensor([[28,  5, 22, 26, 44, 15, 54, 45],
        [16, 22, 17, 32,  0,  0,  0,  0]], device='cuda:0')
tgt_in_b: tensor([[ 1, 45, 54, 15, 44, 26, 22,  5, 28],
        [ 1, 32, 17, 22, 16,  0,  0,  0,  0]], device='cuda:0')
tgt_out_b: tensor([[45, 54, 15, 44, 26, 22,  5, 28,  2],
        [32, 17, 22, 16,  2,  0,  0,  0,  0]], device='cuda:0')


## Model / Loss / Optimizer

In [11]:
d_model = 128
nhead = 4
num_layers = 2
dim_ff = 256

model = Seq2SeqTransformer(
    num_tokens_src=vocab_size,
    num_tokens_tgt=vocab_size,
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_layers,
    num_decoder_layers=num_layers,
    dim_feedforward=dim_ff,
    dropout=0.1,
    max_len=50,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Model parameters: 0.69M


## Simple train loop (Toy example)



In [12]:
def train_step(batch_size=32):
    model.train()
    src, tgt_in, tgt_out = make_batch(batch_size)

    optimizer.zero_grad()
    logits = model(src, tgt_in)              # (B, L, vocab_size)

    B, L, V = logits.shape
    loss = criterion(
        logits.view(B * L, V),
        tgt_out.view(B * L),
    )
    loss.backward()
    optimizer.step()
    return loss.item()

num_steps = 20

for step in range(1, num_steps + 1):
    loss = train_step(batch_size=32)
    if step % 20 == 0:
        print(f"Step {step}/{num_steps} | loss = {loss:.4f}")


Step 20/20 | loss = 3.3424


## Greedy Decoding 함수

In [19]:
def greedy_decode(model, src, max_len=20):
    model.eval()
    with torch.no_grad():
        src_key_padding_mask = (src == PAD)

        src_emb = model.positional_encoding(
            model.src_embed(src) * np.sqrt(model.d_model)
        )
        memory = model.encoder(
            src_emb,
            mask=None,
            src_key_padding_mask=src_key_padding_mask,
        )

        ys = torch.tensor([[SOS]], dtype=torch.long, device=src.device)

        for _ in range(max_len - 1):
            tgt_emb = model.positional_encoding(
                model.tgt_embed(ys) * np.sqrt(model.d_model)
            )

            tgt_mask = model.make_tgt_mask(ys).to(src.device)
            tgt_key_padding_mask = (ys == PAD)

            out = model.decoder(
                tgt_emb,
                memory,
                tgt_mask=tgt_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=src_key_padding_mask,
            )
            logits = model.generator(out)
            next_token = logits[:, -1, :].argmax(-1)

            ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)

            if next_token.item() == EOS:
                break

        return ys.squeeze(0)

src, tgt_in, tgt_out = make_batch(1)
decoded = greedy_decode(model, src)

print("src       :", src[0].tolist())
print("target in :", tgt_in[0].tolist())
print("target out:", tgt_out[0].tolist())
print("decoded   :", decoded.tolist())


src       : [24, 45, 46]
target in : [1, 46, 45, 24]
target out: [46, 45, 24, 2]
decoded   : [1, 45, 2]
